In [1]:
import pandas as pd
file_name = 'Global Food & Nutrition Database 2026 – Health Scores & Allergens.csv'
df = pd.read_csv(file_name)

print(df.columns.tolist())
df.info()

['product_name', 'brands', 'categories', 'ingredients', 'nutriscore_grade', 'nova_group', 'ecoscore_grade', 'allergens', 'energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_soy', 'contains_eggs', 'contains_fish', 'food_type']
<class 'pandas.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        4785 non-null   str    
 1   brands              4751 non-null   str    
 2   categories          4913 non-null   str    
 3   ingredients         4797 non-null   str    
 4   nutriscore_grade    4983 non-null   str    
 5   nova_group          4522 non-null   float64
 6   ecoscore_grade      4984 non-null   str    
 7   allergens           3332 non-null   str    
 8   energy_kcal         4682 non-null 

### Como analizamos en el informe, vemos que los datos como nova_group que puede tomar valores del 1 al 4 es un float64, y que columnas como nutriscore_grade y ecoscore_grade que van de la A a la E (variables cualitativas ordinales) son de tipo object. Transformaremos nova_group a un Int8 para conservar los NaN y las otras dos variables de tipo numéricas para poder trabajarlas correctamente más tarde, ya que casi todas las herramientas de análisis estadístico avanzado en Python lo exigen

In [2]:
df['nova_group'] = df['nova_group'].astype('Int8')

In [3]:
mapeo_ordinal = {
    'A': 5,
    'B': 4,
    'C': 3,
    'D': 2,
    'E': 1
}

df['nutriscore_num'] = df['nutriscore_grade'].str.strip().str.upper().map(mapeo_ordinal).astype('Int8')
df['ecoscore_num'] = df['ecoscore_grade'].str.strip().str.upper().map(mapeo_ordinal).astype('Int8')

df = df.drop(columns=['nutriscore_grade', 'ecoscore_grade'])

In [4]:
## Posteriormente se puede aplicar los siguiente si se queiren visualizar correctamente los datos 
#import matplotlib.pyplot as plt

#df['nutriscore_num'].value_counts().sort_index().plot(kind='bar')

#plt.xticks(
#    ticks=range(5),
#    labels=['E', 'D', 'C', 'B', 'A'],
#    rotation=0
#)

#plt.xlabel('Nutri-Score')
#plt.ylabel('Cantidad de productos')
#plt.title('Distribución del Nutri-Score')
#plt.show()

# El riesgo del falso NAN: 
El único problema con columnas de texto como product name es que el CSV puede tener literalmente la palabra escrita.

- Si la celda del CSV estaba vacía, pandas la lee como el objeto NaN y df.isnull() sí la elimina.

- Si la celda del CSV tenía escrita la palabra "nan", "NaN" o "null" como texto, pandas cree que ese es el nombre del producto. df.isnull() no lo va a contar porque es un String válido.

### Para asegurarnos de que no vamos a tener ese problema: 

In [5]:
import numpy as np

falsos_nulos = ['nan', 'NaN', 'null', 'Null', '']

df = df.replace(falsos_nulos, np.nan)

print(df.isnull().sum())

product_name           212
brands                 246
categories              84
ingredients            200
nova_group             475
allergens             1665
energy_kcal            315
fat_100g               302
saturated_fat_100g     380
carbs_100g             321
sugars_100g            366
fiber_100g            1502
proteins_100g          307
salt_100g              344
sodium_100g            344
contains_gluten          0
contains_dairy           0
contains_nuts            0
contains_soy             0
contains_eggs            0
contains_fish            0
food_type                0
nutriscore_num         550
ecoscore_num          1413
dtype: int64


### Sabemos que todos los productos son empaquetados, por lo que agregaremos a la metadata que solo estamos trabajando con productos de este tipo

In [6]:
# Verifica si absolutamente todos los valores coinciden con el string
if (df['food_type'].str.lower() == 'branded/packaged').all():
    df = df.drop(columns=['food_type'])
    print("Columna 'food type' eliminada: todos los registros eran 'Branded/packaged'.")
else:
    print("La columna 'food type' se conserva: contiene valores distintos a 'Branded/packaged'.")

Columna 'food type' eliminada: todos los registros eran 'Branded/packaged'.


## Tambien debemos considerar eliminar valores mal cargados (fuera de rango)

In [7]:
registros_antes = len(df)

columnas_100g = [col for col in df.columns if col.endswith('_100g')]

df = df[
    (
        df[columnas_100g].ge(0) |
        df[columnas_100g].isna()
    ).all(axis=1) &
    (
        df[columnas_100g].le(100) |
        df[columnas_100g].isna()
    ).all(axis=1)
]

registros_despues = len(df)

print(f"Registros originales: {registros_antes}")
print(f"Registros finales: {registros_despues}")
print(f"Registros eliminados: {registros_antes - registros_despues}")

Registros originales: 4997
Registros finales: 4991
Registros eliminados: 6


# Acerca de los REPETIDOS:

In [8]:
cant_duplicados = df.duplicated().sum()
print("cantidad total de duplicados: " , cant_duplicados)

cantidad total de duplicados:  39


In [9]:
columnas_a_ignorar = ['Name', 'Brand']

columnas_atributos = df.columns.difference(columnas_a_ignorar)

duplicados_por_atributos = df.duplicated(subset=columnas_atributos).sum()
print(f"Duplicados ignorando nombre y marca: {duplicados_por_atributos}")

Duplicados ignorando nombre y marca: 39


### En principio pareceria que tenemos solo 38 filas duplicadas realmente, ya que nos fijamos en este ejemplo en todas las columnas excepto nombre y marca que podrían traer problemas por el idioma

In [10]:

columnas_a_ignorar = ['Name', 'Brand','categories', 'ingredients','allergens']

columnas_atributos = df.columns.difference(columnas_a_ignorar)

duplicados_por_atributos = df.duplicated(subset=columnas_atributos).sum()
print(f"Duplicados ignorando nombre, marca, categorias, ingredientes y alergenos: {duplicados_por_atributos}")

filas_duplicadas = df[df.duplicated(subset=columnas_atributos, keep=False)]

duplicados_agrupados = filas_duplicadas.sort_values(by=list(columnas_atributos))
print(f"Cantidad de filas antes: {len(df)}")

with pd.option_context('display.max_rows', None):
    display(duplicados_agrupados)
    
# para una visualización mas sencilla, usar la linea de abajo y comentar lo de arriba
# display(duplicados_agrupados)

Duplicados ignorando nombre, marca, categorias, ingredientes y alergenos: 74
Cantidad de filas antes: 4991


,product_name,brands,categories,ingredients,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,...,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,nutriscore_num,ecoscore_num
87,Pâte à tartiner noisettes et cacao,Bonne Maman,"en:breakfasts, en:spreads, en:sweet-spreads","sucre, _noisettes_ 20%, huiles végétales (tour...",4,"en:milk, en:nuts",551.0,34.00,5.2,53.0,...,0.10,0.040,False,True,True,False,False,False,1,2
516,Pâte à tartiner noisettes et cacao,Bonne Maman,"en:breakfasts, en:spreads, en:sweet-spreads","Sucre, NOISETTES 20%, huiles végétales (tourne...",4,"en:milk, en:nuts",551.0,34.00,5.2,53.0,...,0.10,0.040,False,True,True,False,False,False,1,2
3543,Petits pains grilles,Carrefour,"en:plant-based-foods-and-beverages, en:plant-b...","Farine de _blé_ 46%, farine de _blé_ complet 4...",4,en:gluten,408.0,10.00,0.9,66.0,...,1.00,0.400,True,False,False,False,False,False,4,5
4025,Petits pains grilles,Carrefour,"en:plant-based-foods-and-beverages, en:plant-b...","Farine de _blé_ 46%, farine de _blé_ complet 4...",4,en:gluten,408.0,10.00,0.9,66.0,...,1.00,0.400,True,False,False,False,False,False,4,5
400,Ratatouille Cuisinée à la Provençal,Cassegrain,"en:plant-based-foods-and-beverages, en:plant-b...",Dés d'aubergines préfrites 25% (aubergines 22%...,3,en:gluten,72.0,4.10,0.6,5.7,...,0.81,0.324,True,False,False,False,False,False,3,4
955,Ratatouille Cuisinée à la Provençal,Cassegrain,"en:plant-based-foods-and-beverages, en:plant-b...",Dés d'aubergines préfrites 25% (aubergines 22%...,3,"en:gluten, en:cereals-with-gluten",72.0,4.10,0.6,5.7,...,0.81,0.324,True,False,False,False,False,False,3,4
546,Our Mature Cheddar,Cathedral City,"en:dairies, en:fermented-foods, en:fermented-m...",cheddar. contains milk.,3,en:milk,416.0,34.90,21.7,0.1,...,1.77,0.708,False,True,False,False,False,False,2,2
1419,Our Mature Cheddar,Cathedral City,"en:dairies, en:fermented-foods, en:fermented-m...",Cheddar Cheese. Contains Milk.,3,en:milk,416.0,34.90,21.7,0.1,...,1.77,0.708,False,True,False,False,False,False,2,2
956,Coca-Cola Zero,Coca-Cola,"en:beverages-and-beverages-preparations, en:be...","Wasser, Kohlensäure, Farbstoff E 150d, Süssung...",4,NaN,0.3,0.00,0.0,0.0,...,0.01,0.004,False,False,False,False,False,False,3,<NA>
3554,Coca-Cola Zero,Coca-Cola,"en:beverages-and-beverages-preparations, en:be...","Agua carbonatada, Colorante e-150d, Edulcorant...",4,en:phenylalanine,0.3,0.00,0.0,0.0,...,0.01,0.004,False,False,False,False,False,False,3,<NA>


### Sin embargo vemos que tenemos muchos repetidos que no eran tenidos en cuenta porque sus ingredientes estaban en dos idiomas distintos o algunos tenían mayúsculas y otros no. Además, vemos una gran cantidad de productos que solo tienen atributos como los ingredientes o sus categorías y no aportan nada acerca de su valor nutricional; por lo que también los descartaremos 

In [11]:
# Eliminar los duplicados basados en los atributos, conservando la primera aparición
df = df.drop_duplicates(subset=columnas_atributos, keep='first')

# Verificar la cantidad de filas que quedaron en el dataset
print(f"Cantidad de filas tras eliminar duplicados: {len(df)}")

Cantidad de filas tras eliminar duplicados: 4917


In [12]:
filas_originales = len(df)

macros_principales = [
    'energy_kcal', 
    'fat_100g', 
    'saturated_fat_100g', 
    'carbs_100g', 
    'sugars_100g', 
    'fiber_100g', 
    'proteins_100g', 
    'salt_100g', 
    'sodium_100g'
]

# se exige al menos 3 datos válidos de los 9
df = df.dropna(subset=macros_principales, thresh=3)

filas_eliminadas = filas_originales - len(df)
print(f"Filas eliminadas por falta de datos nutricionales: {filas_eliminadas}")
print(f"Filas restantes en el dataset: {len(df)}")

Filas eliminadas por falta de datos nutricionales: 231
Filas restantes en el dataset: 4686


### También es interesante plantearse la opción de agregar una columna para poder filtrar la mayor cantidad posible de alimentos segun a que categoría pertenecen, evitando generar sesgos (por ejemplo, contar grasas de productos que en realidad son bebidas y no tienen grasas)

In [13]:
import numpy as np

cats_lower = df['categories'].astype(str).str.lower()

condiciones = [
    # 1. Bebidas (Excluye 'foods-and-beverages' para evitar meter frutos secos y cereales)
    cats_lower.str.contains(r'\b(?:(?<!foods-and-)beverages?|boissons?|bebidas?|juices?|jus|sodas?|waters?|eaux?|aguas?|teas?|thés?|coffees?|cafés?)\b', regex=True),
    
    # 2. Lácteos y derivados
    cats_lower.str.contains(r'\b(?:dairy|dairies|laits?|cheeses?|fromages?|yogurts?|yaourts?|milks?|lacteos?|quesos?)\b', regex=True),
    
    # 3. Snacks dulces, postres y repostería
    cats_lower.str.contains(r'\b(?:snacks?|biscuits?|cakes?|chocolates?|chocolat|cocoa|sweets?|desserts?|biscottes?|confiseries?|galletas?|dulces?)\b', regex=True),
    
    # 4. Comidas preparadas, platos listos y congelados
    cats_lower.str.contains(r'\b(?:meals?|pizzas?|pies?|quiches?|prepared|ready-to-eat|plat-prepare|dishes|dish|salads?|frozen-foods?)\b', regex=True),
    
    # 5. Caldos, sopas, bases para cocinar y deshidratados
    cats_lower.str.contains(r'\b(?:broths?|bouillons?|soups?|soupes?|dried-product|cooking-helper|sopas?|caldos?)\b', regex=True),
    
    # 6. Untables (dulces o salados)
    cats_lower.str.contains(r'\b(?:spreads?|tartines?|pate|paté|pastes?|yeast-extract)\b', regex=True),
    
    # 7. Alimentos a base de plantas, cereales, frutas y verduras
    cats_lower.str.contains(r'\b(?:plant-based|cereals?|breads?|pain|pan|potatoes|potato|legumes?|vegetables?|fruits?|frutas?)\b', regex=True),
    
    # 8. Carnes, pescados, huevos y mariscos
    cats_lower.str.contains(r'\b(?:fish|seafood|meats?|poissons?|viandes?|poultry|pisciculture|eggs?|oeufs?|huevos?|pescados?|carnes?)\b', regex=True),
    
    # 9. Aceites, grasas, salsas y condimentos
    cats_lower.str.contains(r'\b(?:oils?|huiles?|aceites?|sauces?|condiments?|fats?|matiere-grasse)\b', regex=True),
    
    # 10. Suplementos nutricionales y aditivos
    cats_lower.str.contains(r'\b(?:supplements?|dietary|meal-replacement|additives?|yeast)\b', regex=True)
]

categorias_macro = [
    'Bebidas',
    'Lácteos',
    'Snacks y Dulces',
    'Comidas Preparadas',
    'Caldos y Sopas',
    'Untables',
    'Vegetales y Cereales',
    'Carnes y Pescados',
    'Grasas y Salsas',
    'Suplementos y Aditivos'
]

# np.select evalúa en orden. Lo que coincida primero, se asigna.
df['categoria_macro'] = np.select(condiciones, categorias_macro, default='Otros')

# Verificar el nuevo recuento
print(df['categoria_macro'].value_counts())

categoria_macro
Vegetales y Cereales      1499
Snacks y Dulces           1163
Lácteos                    672
Bebidas                    621
Grasas y Salsas            257
Comidas Preparadas         193
Carnes y Pescados          132
Otros                      100
Untables                    22
Caldos y Sopas              14
Suplementos y Aditivos      13
Name: count, dtype: int64


In [14]:
# rellenamos el tratado de las columnas de datos que aportan vbalor energético por cero en lugar de Nan 
columnas = ['energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']
df[columnas] = df[columnas].fillna(0)

In [15]:
# Guardar dataset limpio
df.to_parquet(
    'dataset_limpio.parquet',
    index=False,
    engine='pyarrow'
)